# Intervention comparison on the eight-model panel

Whole-block rotation (the controlled experiments' intervention), branch rotation (the census's),
identity skip, and a rotation-then-inverse plumbing control, block by block, on GPT-2 124M,
Pythia-160M, Mamba-130M, ConvNeXt-tiny, ConvNeXt-base, Llama-3.2-1B, Pythia-6.9B and Qwen2.5-7B.
For each intervention and block: the loss change with a bootstrap CI, and the perturbation
magnitude at the block output relative to the clean residual-stream norm.

Same evaluation data and sizes as the census (30 x 16 x 128 WikiText-103 validation tokens;
CIFAR-100 KL for ConvNeXt). Llama needs an HF token (cell 2). The two 7B models take about
15 min each in float32; the whole panel about 1-1.5 h on an H100/H200. Resume-safe.


In [ ]:
# 1. deps, through the kernel's own interpreter (vast images often have !pip pointing elsewhere)
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "datasets>=3.0", "transformers", "huggingface_hub", "pandas", "pyarrow", "pillow", "scipy"])
import torch, transformers, scipy, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| scipy", scipy.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0); print(p.name, f"{p.total_memory/1e9:.0f} GB")


In [ ]:
# 2. HF token (Llama-3.2-1B is gated) and output dir
import os
os.environ["HF_TOKEN"] = ""            # <- paste your token between the quotes
os.environ["IC_OUT"] = "/workspace/intervention_compare"
os.makedirs(os.environ["IC_OUT"], exist_ok=True)
if not os.environ["HF_TOKEN"]:
    del os.environ["HF_TOKEN"]
print("HF token:", "found" if os.environ.get("HF_TOKEN") else "NONE: Llama will fail, the other seven run")


In [ ]:
%%writefile scramble_census_april_panel.py
"""
Per-layer scrambling census: April-panel completion
====================================================
Runs the same per-layer orthogonal-scrambling assay used for the May census
(delta_l_census_reclass.py, 2026-05-08) on the 15 models of the April 33-model
survey panel that were never scrambled, so the paper can use one panel.

Protocol is unchanged from the May run:
  - Haar-random orthogonal rotation of each sublayer output (attn + mlp for
    decoders; whole block for Mamba and ConvNeXt), dose 1.0, seed 42
  - 30 eval batches x 16 sequences x 128 tokens of WikiText-103 validation
  - dL = mean loss(scrambled) - mean loss(clean); 200-sample bootstrap CI
  - sigma1 profile by finite-difference power iteration on random-token
    inputs (probe dim 32, 15 iterations, 25 inputs), same as the May census
  - R_ex0 (waist-median / edge-median, quadratic vertex) as in the May census
  - output JSON has the same schema as results/delta_l_census_reclass/*.json

Rotation is applied to the *branch* output in every architecture (attn and mlp for
decoders, the SSM mixer for Mamba, the channel-mixing pwconv2 for ConvNeXt), never to
the residual stream itself. Rotating the whole block output, as the May script did for
Mamba, rotates the residual stream and is catastrophic at every layer (and is the
likely reason the May Mamba readouts were unusable). Mamba-1.4B is therefore re-run.

Two additions:
  - the isolated-layer replay for sigma1 captures the layer's real kwargs
    with a forward pre-hook instead of rebuilding rotary embeddings per arch,
    so it works for Phi-2, Llama-3.2, Gemma-2 and Qwen2.5 without special cases
  - ConvNeXt has no token loss, so its primary readout is KL(clean || scrambled)
    over the 1000-class ImageNet head on CIFAR-100 test images (the survey's
    natural-input set), with self-label cross-entropy recorded alongside.
    These are label-free functional readouts, not a variance proxy.

Usage:
  python scramble_census_april_panel.py                # all 15, resume-safe
  python scramble_census_april_panel.py --models gpt2-large microsoft/phi-2
  python scramble_census_april_panel.py --smoke gpt2   # reproduce May gpt2 run
  python scramble_census_april_panel.py --quick ...    # 2 inputs / 2 batches, plumbing check only

Gated models (Llama-3.2, Gemma-2) need HF_TOKEN in the environment and the
licence accepted on the Hub. Qwen2.5-14B in float32 needs ~60 GB of VRAM.
"""

import os, sys, json, time, gc, argparse
import numpy as np

import torch
import torch.nn.functional as F
from scipy import stats

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)
# True float32 everywhere. On Ampere and later (H100, Blackwell) cuDNN convolutions
# default to TF32 (10-bit mantissa), which would corrupt the eps = 1e-3 finite
# differences used for sigma1 on ConvNeXt; matmul TF32 is off by default but is
# forced off here too so the protocol does not depend on the driver's defaults.
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

# ---------------------------------------------------------------- config
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
SEQ_LEN = 128
N_EVAL_BATCHES = 30
EVAL_BATCH_SIZE = 16
N_BOOTSTRAP = 200
DOSE = 1.0
PROBE_DIM = 32
PI_ITERS = 15
N_PROFILE_INPUTS = 25


def log(msg):
    print(msg, flush=True)


def _resolve_hf_token():
    """HF token from the environment, else Colab Secrets, else a cached `huggingface-cli login`."""
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
    except Exception:
        tok = None
    if not tok:
        try:
            from huggingface_hub import get_token
            tok = get_token()
        except Exception:
            tok = None
    if tok:
        os.environ["HF_TOKEN"] = tok
    return tok


_resolve_hf_token()


# The 15 April-panel models without a scrambling run, sorted by size.
APRIL_MISSING = [
    {"hf_id": "facebook/convnext-tiny-224",  "arch": "convnext", "n_layers": 18, "params_M": 28,   "R_ex0_census": 0.232},
    {"hf_id": "facebook/convnext-small-224", "arch": "convnext", "n_layers": 36, "params_M": 50,   "R_ex0_census": 0.569},
    {"hf_id": "facebook/convnext-base-224",  "arch": "convnext", "n_layers": 36, "params_M": 89,   "R_ex0_census": 0.557},
    {"hf_id": "state-spaces/mamba-130m-hf",  "arch": "mamba",    "n_layers": 24, "params_M": 130,  "R_ex0_census": 0.613},
    {"hf_id": "facebook/convnext-large-224", "arch": "convnext", "n_layers": 36, "params_M": 198,  "R_ex0_census": 0.681},
    {"hf_id": "gpt2-large",                  "arch": "gpt2",     "n_layers": 36, "params_M": 774,  "R_ex0_census": 0.890},
    {"hf_id": "meta-llama/Llama-3.2-1B",     "arch": "llama",    "n_layers": 16, "params_M": 1240, "R_ex0_census": 0.720},
    {"hf_id": "gpt2-xl",                     "arch": "gpt2",     "n_layers": 48, "params_M": 1560, "R_ex0_census": 0.951},
    {"hf_id": "google/gemma-2-2b",           "arch": "gemma2",   "n_layers": 26, "params_M": 2600, "R_ex0_census": 0.883},
    {"hf_id": "microsoft/phi-2",             "arch": "phi",      "n_layers": 32, "params_M": 2780, "R_ex0_census": 0.404},
    {"hf_id": "state-spaces/mamba-1.4b-hf",  "arch": "mamba",    "n_layers": 48, "params_M": 1400, "R_ex0_census": 0.956},
    {"hf_id": "state-spaces/mamba-2.8b-hf",  "arch": "mamba",    "n_layers": 64, "params_M": 2800, "R_ex0_census": 1.045},
    {"hf_id": "Qwen/Qwen2.5-3B",             "arch": "qwen",     "n_layers": 36, "params_M": 3090, "R_ex0_census": 0.863},
    {"hf_id": "meta-llama/Llama-3.2-3B",     "arch": "llama",    "n_layers": 28, "params_M": 3210, "R_ex0_census": 0.947},
    {"hf_id": "Qwen/Qwen2.5-7B",             "arch": "qwen",     "n_layers": 28, "params_M": 7620, "R_ex0_census": 0.676},
    {"hf_id": "Qwen/Qwen2.5-14B",            "arch": "qwen",     "n_layers": 48, "params_M": 14770, "R_ex0_census": 0.913},
]

# Reference entry for the smoke test (already in the May census)
PROTOCOL = "branch-rotation-v2"   # files written under an older protocol are re-run

SMOKE = {"gpt2": {"hf_id": "gpt2", "arch": "gpt2", "n_layers": 12, "params_M": 124, "R_ex0_census": 1.082}}

TEXT_ARCHES = ("gpt2", "llama", "gemma2", "phi", "qwen", "mamba", "pythia", "mistral")


# ---------------------------------------------------------------- arch dispatch
def get_layers(model, arch):
    if arch == "gpt2":
        return list(model.transformer.h)
    if arch in ("llama", "gemma2", "phi", "qwen", "mistral"):
        return list(model.model.layers)
    if arch == "pythia":
        return list(model.gpt_neox.layers)
    if arch == "mamba":
        return list(model.backbone.layers)
    if arch == "convnext":
        enc = model.convnext.encoder if hasattr(model, "convnext") else model.encoder
        return [blk for stage in enc.stages for blk in stage.layers]
    raise ValueError(arch)


def get_sublayer_modules(layer, arch):
    """Modules whose outputs are rotated. Same convention as the May census."""
    if arch == "gpt2":
        return (layer.attn, layer.mlp)
    if arch in ("llama", "gemma2", "phi", "qwen", "mistral"):
        return (layer.self_attn, layer.mlp)
    if arch == "pythia":
        return (layer.attention, layer.mlp)
    if arch == "mamba":
        return (layer.mixer,)          # branch (SSM mixer output), before the residual add
    if arch == "convnext":
        return (layer.pwconv2,)        # branch (channel-mixing output, channels-last), before layer-scale and residual add
    raise ValueError(arch)


def sublayer_width(module, layer, arch, d_model):
    """Rotation dimension for a sublayer output (always the last axis of the output)."""
    if arch == "convnext":
        return module.out_features
    return d_model


# ---------------------------------------------------------------- scrambling
def random_orthogonal(d, seed):
    rng = np.random.RandomState(seed)
    H = rng.randn(d, d).astype(np.float32)
    Q, R = np.linalg.qr(H)
    return torch.from_numpy(Q @ np.diag(np.sign(np.diag(R))))


def rotate(x, R, dose):
    """Rotate the last axis of x (the feature axis of every hooked output)."""
    R = R.to(x.device, dtype=x.dtype)
    rot = torch.einsum("ij,...j->...i", R, x)
    return (1 - dose) * x + dose * rot


class ScrambleHookManager:
    def __init__(self):
        self.handles = []

    def register(self, module, R, dose):
        def hook(mod, inp, output):
            if isinstance(output, tuple):
                return (rotate(output[0], R, dose),) + tuple(output[1:])
            return rotate(output, R, dose)
        self.handles.append(module.register_forward_hook(hook))

    def remove_all(self):
        for h in self.handles:
            h.remove()
        self.handles.clear()


# ---------------------------------------------------------------- sigma1 profiling
def make_random_input(arch, inp_idx, vocab_size, image_shape=None):
    torch.manual_seed(SEED + inp_idx * 1000)
    if arch == "convnext":
        return torch.randn(1, *image_shape, device=DEVICE)
    return torch.randint(0, vocab_size, (1, SEQ_LEN), device=DEVICE)


def forward_input(model, arch, inp):
    if arch == "convnext":
        return model(pixel_values=inp)
    return model(input_ids=inp, use_cache=False)


def _strip_cache_kwargs(kw):
    out = {}
    for k, v in kw.items():
        if "past" in k or "cache" in k:
            continue
        out[k] = v
    return out


@torch.no_grad()
def compute_sigma1_profile(model, layers, arch, vocab_size, n_inputs, image_shape=None):
    """Finite-difference power iteration per layer, May-census settings.

    Each layer's real call signature (attention mask, rotary embeddings,
    position ids, ...) is captured with a pre-hook during a normal forward
    and replayed with the perturbed hidden state."""
    model.eval()
    n_layers = len(layers)
    eps = 1e-3
    all_profiles = []

    for inp_idx in range(n_inputs):
        inp = make_random_input(arch, inp_idx, vocab_size, image_shape)
        captured = [None] * n_layers
        hooks = []

        def make_hook(i):
            def fn(mod, args, kwargs):
                captured[i] = (args[0].detach().clone(),
                               tuple(a for a in args[1:]),
                               _strip_cache_kwargs(kwargs))
            return fn

        for i in range(n_layers):
            hooks.append(layers[i].register_forward_pre_hook(make_hook(i), with_kwargs=True))
        forward_input(model, arch, inp)
        for h in hooks:
            h.remove()

        profile = []
        for li in range(n_layers):
            if captured[li] is None:
                profile.append(1.0)
                continue
            x_in, extra_args, kw = captured[li]
            layer = layers[li]

            def layer_fn(x, _l=layer, _a=extra_args, _kw=kw):
                out = _l(x, *_a, **_kw)
                return out[0] if isinstance(out, tuple) else out

            try:
                torch.manual_seed(SEED)
                V = torch.randn(PROBE_DIM, *x_in.shape[1:], device=DEVICE, dtype=x_in.dtype)
                V = V / (V.flatten(1).norm(dim=1).view(-1, *([1] * (V.dim() - 1))) + 1e-12)
                y0 = layer_fn(x_in)
                batched = True
                for _ in range(PI_ITERS):
                    jvp = None
                    if batched:
                        try:               # all probes in one batch; per-sample independent, so identical numerics
                            y1 = layer_fn(x_in + eps * V)
                            if y1.shape[0] == PROBE_DIM:
                                jvp = ((y1 - y0) / eps).detach()
                            else:
                                batched = False
                        except Exception:
                            batched = False
                    if jvp is None:
                        jvps = []
                        for p in range(PROBE_DIM):
                            y1 = layer_fn(x_in + eps * V[p:p + 1])
                            jvps.append(((y1 - y0) / eps).detach())
                        jvp = torch.cat(jvps, dim=0)
                    sig = jvp.float().flatten(1).norm(dim=1)
                    V = (jvp / (sig.view(-1, *([1] * (jvp.dim() - 1))) + 1e-12)).to(x_in.dtype)
                profile.append(float(sig.max().item()))
            except Exception as e:
                log(f"    sigma1 L{li} failed: {e}")
                profile.append(1.0)
        all_profiles.append(profile)
        if (inp_idx + 1) % 5 == 0:
            log(f"    sigma1 profiling: {inp_idx + 1}/{n_inputs} inputs done")

    mean_profile = np.mean(all_profiles, axis=0).tolist()
    log("  sigma1 profile: [" + ", ".join(f"{v:.2f}" for v in mean_profile) + "]")
    return mean_profile


def compute_R_ex0(profile):
    """May-census R_ex0: waist median / edge median about the quadratic vertex, layer 0 excluded."""
    prof = np.array(profile)
    n = len(prof)
    if n < 3:
        return 1.0
    d = np.linspace(0, 1, n)
    X = np.column_stack([d ** 2, d, np.ones(n)])
    beta = np.linalg.lstsq(X, np.log(prof + 1e-12), rcond=None)[0]
    a = beta[0]
    d_star = np.clip(-beta[1] / (2 * a), 0.0, 1.0) if a > 0 else 0.5
    waist = (np.abs(d - d_star) <= 0.20) & (np.arange(n) > 0)
    edge = ((d <= 0.15) | (d >= 0.85)) & (np.arange(n) > 0)
    if waist.sum() == 0 or edge.sum() == 0:
        return 1.0
    e = np.median(prof[edge])
    return float(np.median(prof[waist]) / e) if e > 0 else 1.0


def compute_R_ex0_thirds(profile):
    """April-survey R_ex0: mean(middle third) / mean(edge thirds), layer 0 excluded."""
    prof = np.array(profile)[1:]
    n = len(prof)
    if n < 3:
        return 1.0
    t = n // 3
    mid = prof[t:n - t]
    edge = np.concatenate([prof[:t], prof[n - t:]])
    return float(mid.mean() / edge.mean()) if edge.mean() > 0 else 1.0


# ---------------------------------------------------------------- loading
def _from_pretrained(cls, hf_id, **kw):
    """float32 load; `dtype=` on transformers >= 4.56, `torch_dtype=` before."""
    try:
        return cls.from_pretrained(hf_id, dtype=torch.float32, **kw)
    except TypeError:
        return cls.from_pretrained(hf_id, torch_dtype=torch.float32, **kw)


def load_model(cfg):
    arch, hf_id = cfg["arch"], cfg["hf_id"]
    token = os.environ.get("HF_TOKEN")
    log(f"  Loading {hf_id} ({arch})...")
    if arch == "convnext":
        from transformers import ConvNextForImageClassification, AutoImageProcessor
        model = _from_pretrained(ConvNextForImageClassification, hf_id, token=token).to(DEVICE).eval()
        proc = AutoImageProcessor.from_pretrained(hf_id)
        d_model = None
        return model, proc, d_model
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(hf_id, token=token)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = dict(token=token)
    if arch != "mamba":
        kw["attn_implementation"] = "eager"
    model = _from_pretrained(AutoModelForCausalLM, hf_id, **kw).to(DEVICE).eval()
    d_model = model.config.hidden_size
    return model, tok, d_model


# ---------------------------------------------------------------- data
def _wikitext_validation_texts():
    """WikiText-103 validation split. Reads the parquet file straight from the Hub so the
    run does not depend on the `datasets` builder machinery (which imports scipy.io and
    breaks on some Colab images) or on the bare 'wikitext' id (rejected by recent hubs)."""
    token = os.environ.get("HF_TOKEN")
    try:
        from huggingface_hub import hf_hub_download
        import pandas as pd
        p = hf_hub_download("Salesforce/wikitext", "wikitext-103-raw-v1/validation-00000-of-00001.parquet",
                            repo_type="dataset", token=token)
        return pd.read_parquet(p)["text"].tolist()
    except Exception as e:
        log(f"  parquet download failed ({e}); falling back to datasets")
        from datasets import load_dataset
        return list(load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="validation")["text"])


def _cifar100_test_images(n):
    """CIFAR-100 test images as PIL, via torchvision (direct download) with a datasets fallback."""
    try:
        import torchvision
        ds = torchvision.datasets.CIFAR100(root="./data_cifar100", train=False, download=True)
        return [ds[i][0].convert("RGB") for i in range(n)]
    except Exception as e:
        log(f"  torchvision CIFAR-100 failed ({e}); falling back to datasets")
        from datasets import load_dataset
        ds = load_dataset("uoft-cs/cifar100", split="test")
        return [ds[i]["img"].convert("RGB") for i in range(n)]


def prepare_text_eval(tokenizer, n_seqs):
    texts = _wikitext_validation_texts()
    text = "\n".join(t for t in texts if len(t.strip()) > 50)
    tokens = tokenizer.encode(text)
    log(f"  Tokenized wikitext-103 validation: {len(tokens)} tokens")
    seqs = [tokens[i:i + SEQ_LEN] for i in range(0, len(tokens) - SEQ_LEN, SEQ_LEN)][:n_seqs]
    ids = torch.tensor(seqs, dtype=torch.long)
    log(f"  Eval data: {ids.shape[0]} sequences of length {SEQ_LEN}")
    return ids


def prepare_image_eval(proc, n_images):
    """CIFAR-100 test images (the April survey's natural-input set), model preprocessing."""
    imgs = _cifar100_test_images(n_images)
    px = proc(images=imgs, return_tensors="pt")["pixel_values"]
    log(f"  Eval data: {px.shape[0]} CIFAR-100 test images -> {tuple(px.shape[1:])}")
    return px


# ---------------------------------------------------------------- assay
@torch.no_grad()
def run_perlayer_scrambling(model, layers, arch, d_model, eval_data, sigma1_profile, n_eval):
    n_layers = len(layers)
    is_vision = arch == "convnext"

    R = {}
    for li in range(n_layers):
        subs = get_sublayer_modules(layers[li], arch)
        for si, sm in enumerate(subs):
            w = sublayer_width(sm, layers[li], arch, d_model)
            R[(li, si)] = random_orthogonal(w, SEED + li * 100 + si * 10)

    clean_logp = [None] * n_eval   # vision only: cached clean log-probs per batch

    def eval_pass(target_layers):
        mgr = ScrambleHookManager()
        try:
            for li in target_layers:
                for si, sm in enumerate(get_sublayer_modules(layers[li], arch)):
                    mgr.register(sm, R[(li, si)], DOSE)
            losses, kls, selfce = [], [], []
            for b in range(n_eval):
                batch = eval_data[b * EVAL_BATCH_SIZE:(b + 1) * EVAL_BATCH_SIZE].to(DEVICE)
                if is_vision:
                    logits = model(pixel_values=batch).logits.float()
                    logp = F.log_softmax(logits, dim=-1)
                    if clean_logp[b] is None:
                        clean_logp[b] = logp.detach().clone()
                    cl = clean_logp[b]
                    kl = (cl.exp() * (cl - logp)).sum(-1).mean().item()
                    ce = F.nll_loss(logp, cl.argmax(-1)).item()
                    losses.append(kl); kls.append(kl); selfce.append(ce)
                else:
                    out = model(input_ids=batch, labels=batch, use_cache=False)
                    losses.append(out.loss.item())
            return losses, kls, selfce
        finally:
            mgr.remove_all()

    base, base_kl, base_ce = eval_pass([])
    baseline = float(np.mean(base))
    base_arr = np.array(base)
    log(f"  Baseline {'KL' if is_vision else 'loss'}: {baseline:.4f}" +
        (f"  (self-label CE {np.mean(base_ce):.4f})" if is_vision else ""))

    dl, ci, extra = [], [], []
    for li in range(n_layers):
        ls, kl, ce = eval_pass([li])
        arr = np.array(ls)
        delta = float(arr.mean() - baseline)
        rng = np.random.RandomState(SEED + li)
        boots = []
        for _ in range(N_BOOTSTRAP):
            idx = rng.randint(0, n_eval, n_eval)
            boots.append(arr[idx].mean() - base_arr[idx].mean())
        lo, hi = float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))
        dl.append(delta); ci.append([lo, hi])
        if is_vision:
            extra.append({"kl": float(np.mean(kl)), "selflabel_ce": float(np.mean(ce)),
                          "delta_selflabel_ce": float(np.mean(ce) - np.mean(base_ce))})
        s1 = sigma1_profile[li] if li < len(sigma1_profile) else 0
        log(f"    L{li:2d}: dL={delta:+.4f} [{lo:+.4f},{hi:+.4f}]  sigma1={s1:.2f}")

    d = np.array(dl); s = np.array(sigma1_profile[:n_layers])
    rho_all, p_all = stats.spearmanr(d, s) if n_layers > 1 else (0, 1)
    rho_ex0, p_ex0 = stats.spearmanr(d[1:], s[1:]) if n_layers > 2 else (0, 1)
    r_log, p_log = stats.pearsonr(np.log(np.clip(d[1:], 1e-6, None)), np.log(s[1:] + 1e-12)) if n_layers > 2 else (0, 1)
    log(f"  Spearman rho (excl. L0): {rho_ex0:.3f} (p={p_ex0:.4f})")

    out = {
        "baseline": baseline,
        "delta_L_profile": dl,
        "delta_L_ci": ci,
        "sigma1_profile": list(map(float, s)),
        "spearman_all": {"rho": float(rho_all), "p": float(p_all)},
        "spearman_ex0": {"rho": float(rho_ex0), "p": float(p_ex0)},
        "pearson_log_ex0": {"r": float(r_log), "p": float(p_log)},
    }
    if is_vision:
        out["readout"] = "KL(clean||scrambled) over ImageNet-1k head, CIFAR-100 test images; delta_L_profile is delta-KL in nats"
        out["baseline_selflabel_ce"] = float(np.mean(base_ce))
        out["vision_extra"] = extra
    else:
        out["readout"] = "next-token cross-entropy, WikiText-103 validation"
    return out


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="*", help="hf_ids to run (default: all 15)")
    ap.add_argument("--smoke", nargs="*", help="run reference models (gpt2) to compare with the May census")
    ap.add_argument("--quick", action="store_true", help="2 profiling inputs, 2 eval batches (plumbing check only)")
    ap.add_argument("--results-dir", default="results/delta_l_census_reclass")
    ap.add_argument("--no-skip", action="store_true")
    ap.add_argument("--zip", action="store_true", help="zip the results dir at the end (Colab download)")
    # parse_known_args so the script also runs pasted into a Jupyter/Colab cell
    # (the kernel adds "-f kernel.json" to sys.argv). In a notebook, set options
    # through environment variables instead: SCRAMBLE_MODELS="a b", SCRAMBLE_SMOKE="gpt2",
    # SCRAMBLE_QUICK=1, SCRAMBLE_RESULTS_DIR=..., SCRAMBLE_NO_SKIP=1, SCRAMBLE_ZIP=1
    args, _unknown = ap.parse_known_args()
    env = os.environ.get
    if env("SCRAMBLE_MODELS"): args.models = env("SCRAMBLE_MODELS").split()
    if env("SCRAMBLE_SMOKE"): args.smoke = env("SCRAMBLE_SMOKE").split()
    if env("SCRAMBLE_QUICK"): args.quick = True
    if env("SCRAMBLE_RESULTS_DIR"): args.results_dir = env("SCRAMBLE_RESULTS_DIR")
    if env("SCRAMBLE_NO_SKIP"): args.no_skip = True
    if env("SCRAMBLE_ZIP"): args.zip = True

    n_inputs = 2 if args.quick else N_PROFILE_INPUTS
    n_eval = 2 if args.quick else N_EVAL_BATCHES
    os.makedirs(args.results_dir, exist_ok=True)
    suffix = "_quick" if args.quick else ""

    todo = list(APRIL_MISSING)
    if args.smoke:
        todo = [SMOKE[m] for m in args.smoke]
        suffix = "_smoke" + suffix
    if args.models:
        todo = [c for c in todo if c["hf_id"] in args.models]
        missing = set(args.models) - {c["hf_id"] for c in todo}
        if missing:
            log(f"unknown models: {missing}")

    log(f"Device: {DEVICE}; models: {[c['hf_id'] for c in todo]}")
    if os.environ.get("HF_TOKEN"):
        log("HF token: found")
    else:
        log("WARNING: no HF token (env HF_TOKEN, Colab secret HF_TOKEN, or huggingface-cli login). "
            "meta-llama/Llama-3.2-* and google/gemma-2-2b are gated and will fail.")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        vram = getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)
        cap = torch.cuda.get_device_capability(0)
        log(f"GPU: {torch.cuda.get_device_name()} ({vram / 1e9:.0f} GB, sm_{cap[0]}{cap[1]}); "
            f"torch {torch.__version__}; TF32 conv={torch.backends.cudnn.allow_tf32} matmul={torch.backends.cuda.matmul.allow_tf32}")
        # fail early if this torch build has no kernels for the GPU (e.g. Blackwell sm_120 needs a cu128+ build)
        torch.ones(2, 2, device="cuda") @ torch.ones(2, 2, device="cuda")

    for cfg in todo:
        hf_id, arch = cfg["hf_id"], cfg["arch"]
        fpath = os.path.join(args.results_dir, f"perlayer_{hf_id.replace('/', '_').lower()}{suffix}.json")
        if os.path.exists(fpath) and not args.no_skip:
            try:
                old = json.load(open(fpath)).get("protocol")
            except Exception:
                old = None
            if old == PROTOCOL or arch not in ("mamba", "convnext"):
                log(f"[SKIP] {hf_id} exists")          # transformer files from the earlier run are unchanged by v2
                continue
            log(f"[REDO] {hf_id}: existing file is protocol {old!r}, current is {PROTOCOL!r}")
        log("\n" + "-" * 78 + f"\n{hf_id} ({arch}, {cfg['n_layers']}L, {cfg['params_M']}M)\n" + "-" * 78)
        t0 = time.time()
        model = tok = None
        try:
            model, tok, d_model = load_model(cfg)
            layers = get_layers(model, arch)
            assert len(layers) == cfg["n_layers"], f"layer count {len(layers)} != {cfg['n_layers']}"
            if arch == "convnext":
                eval_data = prepare_image_eval(tok, n_eval * EVAL_BATCH_SIZE)
                image_shape = tuple(eval_data.shape[1:])
                vocab = None
            else:
                eval_data = prepare_text_eval(tok, n_eval * EVAL_BATCH_SIZE)
                image_shape = None
                vocab = model.config.vocab_size

            log(f"  sigma1 profile ({n_inputs} inputs x {PI_ITERS} iters x {PROBE_DIM} probes)...")
            t1 = time.time()
            s1 = compute_sigma1_profile(model, layers, arch, vocab, n_inputs, image_shape)
            R_live = compute_R_ex0(s1)
            R_thirds = compute_R_ex0_thirds(s1)
            log(f"  sigma1 done ({time.time() - t1:.0f}s): R_ex0 vertex={R_live:.3f}, thirds={R_thirds:.3f} (survey {cfg['R_ex0_census']:.3f})")

            res = run_perlayer_scrambling(model, layers, arch, d_model, eval_data, s1, n_eval)
            res.update({"hf_id": hf_id, "arch": arch, "n_layers": cfg["n_layers"],
                        "d_model": d_model, "params_M": cfg["params_M"],
                        "R_ex0_census": cfg["R_ex0_census"], "R_ex0_live": R_live,
                        "R_ex0_thirds_live": R_thirds, "elapsed_seconds": time.time() - t0,
                        "panel": "april-33", "quick": bool(args.quick), "protocol": PROTOCOL})
            with open(fpath, "w") as f:
                json.dump(res, f, indent=2)
            log(f"  saved {fpath} ({time.time() - t0:.0f}s)")
        except Exception as e:
            import traceback; traceback.print_exc()
            log(f"  FAILED {hf_id}: {e}")
        finally:
            del model, tok
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if args.zip:
        import shutil
        out = shutil.make_archive("census_results", "zip", args.results_dir)
        log(f"zipped results -> {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile intervention_compare.py
#!/usr/bin/env python3
"""
Intervention comparison on eight pretrained models (18 Sept 2026)
================================================================
The controlled experiments rotate a block's OUTPUT (the residual state); the pretrained census
rotates each sublayer's BRANCH output and leaves the residual stream unrotated. Those are
different interventions. This script applies both, plus identity skipping, block by block, on the
eight-model protocol panel, and records for each the loss change AND the actual perturbation
magnitude at the block output relative to the clean residual-stream norm, so equal rotation dose
is no longer assumed to be equal intervention strength.

Per block l, four passes over the evaluation set:
  block   Q [h_l + F_l(h_l)]          Haar rotation of the block output. Residual norm preserved.
  branch  h_l + Q F_l(h_l)            Haar rotation of each sublayer's output (the census rule).
                                      Branch norm preserved; residual norm not.
  skip    h_l                          block output replaced by its input (computation removed,
                                      alignment preserved).
  plumb   Q at block l output, Q^T at block l+1 input.  Must reproduce the clean loss; a check
                                      that the hooks act where they claim to (blocks 0..L-2).

Readout: next-token cross-entropy on WikiText-103 validation (30 x 16 x 128 tokens) for language
models; KL(clean || perturbed) over the ImageNet head on CIFAR-100 for ConvNeXt. Same data and
sizes as the census. Magnitude: ||h'_{l+1} - h_{l+1}|| / ||h_{l+1}|| on the first evaluation
batch, averaged over tokens, where h_{l+1} is the block output on the clean pass.

Reuses scramble_census_april_panel.py (must sit beside this file) for model loading, layer and
sublayer lookup, Haar rotations and the evaluation data. One JSON per model, resume-safe.

    HF_TOKEN=... python3 intervention_compare.py --out results/intervention_compare
"""

import os, sys, json, time, datetime, argparse
import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import scramble_census_april_panel as sc

PANEL = [
    ("gpt2",                        "gpt2"),
    ("EleutherAI/pythia-160m",      "pythia"),
    ("state-spaces/mamba-130m-hf",  "mamba"),
    ("facebook/convnext-tiny-224",  "convnext"),
    ("facebook/convnext-base-224",  "convnext"),
    ("meta-llama/Llama-3.2-1B",     "llama"),
    ("EleutherAI/pythia-6.9b",      "pythia"),
    ("Qwen/Qwen2.5-7B",             "qwen"),
]
DEVICE = sc.DEVICE


def log(m=""):
    print(f"[{time.strftime('%H:%M:%S')}] {m}" if m else "", flush=True)


# ---------------------------------------------------------------- hooks
def _hidden_of(output):
    return output[0] if isinstance(output, (tuple, list)) else output


def _with_hidden(output, h):
    return (h,) + tuple(output[1:]) if isinstance(output, (tuple, list)) else h


def _input_hidden(args, kwargs):
    return args[0] if len(args) else kwargs["hidden_states"]


def rot(x, Q, axis):
    """Rotate x along `axis` by Q (d x d)."""
    Q = Q.to(x.device, dtype=x.dtype)
    x = x.movedim(axis, -1)
    y = torch.einsum("ij,...j->...i", Q, x)
    return y.movedim(-1, axis)


class Hooks:
    def __init__(self):
        self.h = []

    def add(self, handle):
        self.h.append(handle)

    def clear(self):
        for x in self.h:
            x.remove()
        self.h.clear()


def block_axis(arch):
    return 1 if arch == "convnext" else -1        # ConvNeXt block outputs are (N, C, H, W)


# ---------------------------------------------------------------- one model
def run_model(hf_id, arch, args):
    model, tok_or_proc, d_model = sc.load_model({"hf_id": hf_id, "arch": arch})
    layers = sc.get_layers(model, arch)
    L = len(layers)
    is_vision = arch == "convnext"
    n_eval = args.n_eval
    if is_vision:
        data = sc.prepare_image_eval(tok_or_proc, n_eval * sc.EVAL_BATCH_SIZE)
    else:
        data = sc.prepare_text_eval(tok_or_proc, n_eval * sc.EVAL_BATCH_SIZE)
    bs = sc.EVAL_BATCH_SIZE
    log(f"  {L} blocks; {n_eval} eval batches of {bs}")

    # block-output width per block (ConvNeXt changes width across stages)
    widths = []
    with torch.no_grad():
        caps = {}
        hk = Hooks()
        for li in range(L):
            hk.add(layers[li].register_forward_hook(lambda m, a, o, li=li: caps.__setitem__(li, _hidden_of(o).detach())))
        b0 = data[:bs].to(DEVICE)
        sc.forward_input(model, arch, b0)
        hk.clear()
    for li in range(L):
        widths.append(caps[li].shape[block_axis(arch)])
    clean_out0 = {li: caps[li].float() for li in range(L)}     # clean block outputs, batch 0
    del caps

    Qb = {li: sc.random_orthogonal(widths[li], sc.SEED + 1000 + li) for li in range(L)}
    Qs = {}
    for li in range(L):
        for si, sm in enumerate(sc.get_sublayer_modules(layers[li], arch)):
            Qs[(li, si)] = sc.random_orthogonal(sc.sublayer_width(sm, layers[li], arch, d_model), sc.SEED + li * 100 + si * 10)

    clean_logp = [None] * n_eval

    @torch.no_grad()
    def eval_pass(setup, capture_block=None):
        """setup(hk) registers the intervention hooks. Returns (per-batch losses, magnitude at capture_block on batch 0)."""
        hk = Hooks()
        mag = None
        try:
            setup(hk)
            store = {}
            if capture_block is not None:
                hk.add(layers[capture_block].register_forward_hook(lambda m, a, o: store.__setitem__("h", _hidden_of(o).detach().float())))
            losses = []
            for b in range(n_eval):
                batch = data[b * bs:(b + 1) * bs].to(DEVICE)
                if is_vision:
                    logp = F.log_softmax(model(pixel_values=batch).logits.float(), dim=-1)
                    if clean_logp[b] is None:
                        clean_logp[b] = logp.clone()
                    cl = clean_logp[b]
                    losses.append((cl.exp() * (cl - logp)).sum(-1).mean().item())
                else:
                    losses.append(model(input_ids=batch, labels=batch, use_cache=False).loss.item())
                if b == 0 and capture_block is not None:
                    h1, h0 = store["h"], clean_out0[capture_block]
                    ax = block_axis(arch)
                    diff = (h1 - h0).movedim(ax, -1).flatten(0, -2).norm(dim=-1)
                    base = h0.movedim(ax, -1).flatten(0, -2).norm(dim=-1)
                    mag = float((diff / base.clamp_min(1e-12)).mean())
            return losses, mag
        finally:
            hk.clear()

    base, _ = eval_pass(lambda hk: None)
    baseline = float(np.mean(base)); base_arr = np.array(base)
    log(f"  baseline {'KL' if is_vision else 'loss'} {baseline:.4f}")

    def mk_block(li):
        def setup(hk):
            hk.add(layers[li].register_forward_hook(lambda m, a, o: _with_hidden(o, rot(_hidden_of(o), Qb[li], block_axis(arch)))))
        return setup

    def mk_branch(li):
        def setup(hk):
            for si, sm in enumerate(sc.get_sublayer_modules(layers[li], arch)):
                hk.add(sm.register_forward_hook(lambda m, a, o, Q=Qs[(li, si)]: _with_hidden(o, rot(_hidden_of(o), Q, -1))))
        return setup

    def mk_skip(li):
        def setup(hk):
            def fh(m, a, kw, o):
                return _with_hidden(o, _input_hidden(a, kw))
            hk.add(layers[li].register_forward_hook(fh, with_kwargs=True))
        return setup

    def mk_plumb(li):
        def setup(hk):
            hk.add(layers[li].register_forward_hook(lambda m, a, o: _with_hidden(o, rot(_hidden_of(o), Qb[li], block_axis(arch)))))
            QT = Qb[li].T.contiguous()

            def pre(m, a, kw):
                if len(a):
                    return (rot(a[0], QT, block_axis(arch)),) + tuple(a[1:]), kw
                kw = dict(kw); kw["hidden_states"] = rot(kw["hidden_states"], QT, block_axis(arch)); return a, kw
            hk.add(layers[li + 1].register_forward_pre_hook(pre, with_kwargs=True))
        return setup

    results = {k: [] for k in ("block", "branch", "skip", "plumb")}
    mags = {k: [] for k in ("block", "branch", "skip")}
    t0 = time.time()
    for li in range(L):
        row = {}
        for name, mk in (("block", mk_block), ("branch", mk_branch), ("skip", mk_skip)):
            ls, mag = eval_pass(mk(li), capture_block=li)
            arr = np.array(ls); d = float(arr.mean() - baseline)
            rng = np.random.RandomState(sc.SEED + li)
            boots = [arr[i].mean() - base_arr[i].mean() for i in (rng.randint(0, n_eval, n_eval) for _ in range(sc.N_BOOTSTRAP))]
            results[name].append(dict(block=li, dL=d, ci=[float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))], magnitude=mag))
            mags[name].append(mag); row[name] = (d, mag)
        if li < L - 1 and widths[li] == widths[li + 1]:
            # the control needs block l+1 to receive block l's output unchanged; at a ConvNeXt
            # stage boundary a downsampling layer sits between them, so the check is skipped there
            ls, _ = eval_pass(mk_plumb(li))
            dp = float(np.mean(ls) - baseline); results["plumb"].append(dict(block=li, dL=dp))
        else:
            dp = float("nan")
        log(f"    L{li:2d}  block dL {row['block'][0]:+.4f} (|dh|/|h| {row['block'][1]:.2f})   branch {row['branch'][0]:+.4f} ({row['branch'][1]:.2f})   "
            f"skip {row['skip'][0]:+.4f} ({row['skip'][1]:.2f})   plumb {dp:+.1e}   [{time.time() - t0:.0f} s]")

    def thirds(v):
        v = np.asarray(v, float)[1:L - 1]; n = len(v) // 3
        return dict(early=float(v[:n].mean()), waist=float(v[n:len(v) - n].mean()), late=float(v[len(v) - n:].mean()))
    summary = {}
    for name in ("block", "branch", "skip"):
        dl = [r["dL"] for r in results[name]]; t = thirds(dl)
        summary[name] = dict(**t, waist_over_edge=t["waist"] / max(t["early"], t["late"], 1e-12),
                             magnitude_median=float(np.nanmedian(mags[name])), magnitude_range=[float(np.nanmin(mags[name])), float(np.nanmax(mags[name]))])
    plumb_max = float(np.max(np.abs([r["dL"] for r in results["plumb"]]))) if results["plumb"] else None
    from scipy import stats
    rho_bb = stats.spearmanr([r["dL"] for r in results["block"]][1:L - 1], [r["dL"] for r in results["branch"]][1:L - 1])[0]
    rho_bs = stats.spearmanr([r["dL"] for r in results["block"]][1:L - 1], [r["dL"] for r in results["skip"]][1:L - 1])[0]
    log(f"  waist/edge  block {summary['block']['waist_over_edge']:.3f}  branch {summary['branch']['waist_over_edge']:.3f}  skip {summary['skip']['waist_over_edge']:.3f}"
        f"  | median |dh|/|h|  block {summary['block']['magnitude_median']:.2f} branch {summary['branch']['magnitude_median']:.2f} skip {summary['skip']['magnitude_median']:.2f}"
        f"  | plumbing max |dL| {plumb_max:.1e}  | Spearman block~branch {rho_bb:.2f}, block~skip {rho_bs:.2f}")
    out = dict(hf_id=hf_id, arch=arch, n_layers=L, baseline=baseline, n_eval_batches=n_eval, batch=bs, seq_len=sc.SEQ_LEN,
               readout="KL(clean||perturbed), ImageNet head, CIFAR-100" if is_vision else "next-token CE, WikiText-103 validation",
               widths=widths, per_block=results, summary=summary, plumbing_max_abs_dL=plumb_max,
               spearman_block_vs_branch=float(rho_bb), spearman_block_vs_skip=float(rho_bs),
               time_s=time.time() - t0, gpu=torch.cuda.get_device_name() if DEVICE == "cuda" else "cpu",
               torch=torch.__version__, date=datetime.datetime.now().isoformat(timespec="seconds"))
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return out


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="*", default=None)
    ap.add_argument("--out", default=os.environ.get("IC_OUT", "results/intervention_compare"))
    ap.add_argument("--n-eval", type=int, default=int(os.environ.get("IC_N_EVAL", sc.N_EVAL_BATCHES)))
    ap.add_argument("--no-skip", action="store_true")
    args, _ = ap.parse_known_args(argv)
    if os.environ.get("IC_MODELS"):
        args.models = os.environ["IC_MODELS"].split()
    sc._resolve_hf_token()
    os.makedirs(args.out, exist_ok=True)
    todo = [(h, a) for h, a in PANEL if not args.models or h in args.models]
    log(f"intervention comparison on {DEVICE}: {[h for h, _ in todo]}")
    log(f"HF token: {'found' if os.environ.get('HF_TOKEN') else 'none (Llama will fail)'}")
    for k, (hf_id, arch) in enumerate(todo):
        path = os.path.join(args.out, f"ic_{hf_id.replace('/', '_')}.json")
        if os.path.exists(path) and not args.no_skip:
            log(f"[{k + 1}/{len(todo)}] {hf_id}: done, skipping"); continue
        log(f"\n[{k + 1}/{len(todo)}] {hf_id} ({arch})")
        try:
            res = run_model(hf_id, arch, args)
        except Exception as e:
            import traceback; traceback.print_exc(); log(f"  FAILED {hf_id}: {e}"); continue
        tmp = path + ".tmp"
        json.dump(res, open(tmp, "w"), indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
        os.replace(tmp, path)
    summary(args.out)


def summary(out):
    rows = [json.load(open(os.path.join(out, f))) for f in sorted(os.listdir(out)) if f.startswith("ic_") and f.endswith(".json")]
    if not rows:
        return
    print("\n%-28s %8s | %-22s | %-22s | %-22s | %9s" % ("model", "base", "block  w/e  |dh|/|h|", "branch w/e  |dh|/|h|", "skip   w/e  |dh|/|h|", "plumb max"))
    for r in rows:
        s = r["summary"]
        print("%-28s %8.3f | %6.3f %6.2f      | %6.3f %6.2f      | %6.3f %6.2f      | %9.1e" % (
            r["hf_id"], r["baseline"], s["block"]["waist_over_edge"], s["block"]["magnitude_median"],
            s["branch"]["waist_over_edge"], s["branch"]["magnitude_median"], s["skip"]["waist_over_edge"], s["skip"]["magnitude_median"],
            r["plumbing_max_abs_dL"] or float("nan")))


if __name__ == "__main__":
    main()


In [ ]:
# 5. run (resume-safe; re-run after any restart)
%run intervention_compare.py

In [ ]:
# 6. table, safe to re-run
import json, glob
rows = [json.load(open(f)) for f in sorted(glob.glob("/workspace/intervention_compare/ic_*.json"))]
print(f"{len(rows)}/8 done")
print("%-28s %8s | %-20s | %-20s | %-20s | %9s | rho b~br b~sk" % ("model", "base", "block  w/e  |dh|/|h|", "branch w/e |dh|/|h|", "skip   w/e |dh|/|h|", "plumb max"))
for r in rows:
    s = r["summary"]
    print("%-28s %8.3f | %6.3f %6.2f       | %6.3f %6.2f       | %6.3f %6.2f       | %9.1e | %5.2f %5.2f" % (
        r["hf_id"], r["baseline"], s["block"]["waist_over_edge"], s["block"]["magnitude_median"],
        s["branch"]["waist_over_edge"], s["branch"]["magnitude_median"], s["skip"]["waist_over_edge"], s["skip"]["magnitude_median"],
        r["plumbing_max_abs_dL"], r["spearman_block_vs_branch"], r["spearman_block_vs_skip"]))

In [ ]:
# 7. zip
import shutil
z = shutil.make_archive("/workspace/intervention_compare", "zip", "/workspace/intervention_compare")
print(z)
from IPython.display import FileLink; FileLink(z)